# Trabalho Prático 05 — Agentes Inteligentes para Othello

**Disciplina:** GCC128 — Inteligência Artificial  
**Tecnologias:** Python 3, LangChain, LangGraph e Groq

O notebook implementa Othello, um agente heurístico e um agente LLM com dois modos de autonomia.

## 1. Configuração do ambiente

Na raiz do projeto, crie o ambiente virtual e instale as dependências:

**Windows (PowerShell):**
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
```

**Ubuntu:**
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

No VS Code, abra o notebook, clique em **Selecionar Kernel** e escolha o Python do ambiente `.venv`.


### 1.1 Verificar as dependências

A célula importa as bibliotecas e exibe suas versões.

In [140]:
import sys
from importlib.metadata import version

import pygame
import langchain
import langgraph
import langchain_groq

print("Python:", sys.version.split()[0])
for pacote in ("pygame", "langchain", "langgraph", "langchain-groq"):
    print(f"{pacote}: {version(pacote)}")
print("Ambiente configurado. Nenhum agente foi executado.")

Python: 3.12.3
pygame: 2.6.1
langchain: 1.5.1
langgraph: 1.2.22
langchain-groq: 1.2.0
Ambiente configurado. Nenhum agente foi executado.


## 2. Motor e estado da partida

O motor implementa as regras do tabuleiro 8×8. O estado guarda o tabuleiro, o jogador atual e o histórico.

### 2.1 Tabuleiro e representação

In [141]:
# Representação das casas
VAZIO = 0
PRETO = 1
BRANCO = -1

TAMANHO = 8

def criar_tabuleiro():
    tabuleiro = [
        [VAZIO for _ in range(TAMANHO)]
        for _ in range(TAMANHO)
    ]

    # Posições iniciais das peças
    tabuleiro[3][3] = BRANCO
    tabuleiro[3][4] = PRETO
    tabuleiro[4][3] = PRETO
    tabuleiro[4][4] = BRANCO

    return tabuleiro


def exibir_tabuleiro(tabuleiro):
    simbolos = {
        VAZIO: ".",
        PRETO: "P",
        BRANCO: "B"
    }

    print("    " + " ".join(str(i) for i in range(TAMANHO)))

    for i, linha in enumerate(tabuleiro):
        print(f"{i}   " + " ".join(simbolos[peca] for peca in linha))

### 2.2 Capturas e movimentos legais

In [142]:
# Oito direções possíveis: horizontal, vertical e diagonais
DIRECOES = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1)
]


def pecas_capturadas(tabuleiro, linha, coluna, jogador):
    if not (0 <= linha < TAMANHO and 0 <= coluna < TAMANHO):
        return []

    if tabuleiro[linha][coluna] != VAZIO:
        return []

    capturadas = []

    for dl, dc in DIRECOES:
        l, c = linha + dl, coluna + dc
        caminho = []

        # Percorre as peças adversárias nesta direção
        while (
            0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == -jogador
        ):
            caminho.append((l, c))
            l += dl
            c += dc

        # A sequência só é válida se terminar em uma peça própria
        if (
            caminho
            and 0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == jogador
        ):
            capturadas.extend(caminho)

    return capturadas


def movimentos_validos(tabuleiro, jogador):
    movimentos = []

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):
            if pecas_capturadas(tabuleiro, linha, coluna, jogador):
                movimentos.append((linha, coluna))

    return movimentos

### 2.3 Aplicação de jogadas

In [143]:
def fazer_jogada(tabuleiro, linha, coluna, jogador):
    # Identifica as peças que serão capturadas
    capturadas = pecas_capturadas(
        tabuleiro, linha, coluna, jogador
    )

    # Impede movimentos inválidos
    if not capturadas:
        return False

    # Coloca a peça do jogador
    tabuleiro[linha][coluna] = jogador

    # Inverte as peças adversárias capturadas
    for l, c in capturadas:
        tabuleiro[l][c] = jogador

    return True

### 2.4 Alternância e passagem de turno

In [144]:
def proximo_turno(tabuleiro, jogador_atual):
    adversario = -jogador_atual

    # Verifica se o adversário pode jogar
    if movimentos_validos(tabuleiro, adversario):
        return adversario

    # Se o adversário não puder, o jogador atual continua
    if movimentos_validos(tabuleiro, jogador_atual):
        return jogador_atual

    # Nenhum jogador possui movimentos válidos
    return None

### 2.5 Placar e encerramento

In [145]:
def contar_pecas(tabuleiro):
    pretas = sum(linha.count(PRETO) for linha in tabuleiro)
    brancas = sum(linha.count(BRANCO) for linha in tabuleiro)

    return pretas, brancas


def jogo_terminou(tabuleiro):
    return (
        not movimentos_validos(tabuleiro, PRETO)
        and not movimentos_validos(tabuleiro, BRANCO)
    )


def vencedor_por_placar(pretas, brancas):
    if pretas > brancas:
        return PRETO
    if brancas > pretas:
        return BRANCO
    return VAZIO  # Empate


def obter_vencedor(tabuleiro):
    if not jogo_terminou(tabuleiro):
        return None
    return vencedor_por_placar(*contar_pecas(tabuleiro))

### 2.6 Estado inicial e histórico

In [146]:
def iniciar_partida():
    return {
        "tabuleiro": criar_tabuleiro(),
        "jogador_atual": PRETO,
        "historico": []
    }


# Estado compartilhado entre as ferramentas e os agentes.
estado_jogo = iniciar_partida()

In [147]:
def obter_historico():
    historico = estado_jogo["historico"]

    if not historico:
        return "Nenhuma jogada realizada."

    jogadas = []
    for numero, jogada in enumerate(historico, start=1):
        jogador = "PRETO" if jogada["player"] == PRETO else "BRANCO"
        jogadas.append(
            f"{numero}. {jogador}: "
            f"linha {jogada['row']}, coluna {jogada['col']}"
        )

    return "\n".join(jogadas)

## 3. Algoritmos de estratégia

### 3.1 Heurística posicional

A avaliação soma os pesos das peças do jogador e subtrai os pesos das peças adversárias.

In [148]:
# Pesos estratégicos das posições do tabuleiro
PESOS_POSICAO = [
    [100, -20,  10,   5,   5,  10, -20, 100],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [100, -20,  10,   5,   5,  10, -20, 100]
]

In [149]:
def avaliar_tabuleiro(tabuleiro, jogador):
    pontuacao = 0

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):

            peca = tabuleiro[linha][coluna]

            if peca == jogador:
                pontuacao += PESOS_POSICAO[linha][coluna]

            elif peca == -jogador:
                pontuacao -= PESOS_POSICAO[linha][coluna]

    return pontuacao

### 3.2 Simulação e escolha da jogada

A heurística avalia as jogadas legais em cópias do tabuleiro. Em caso de empate, escolhe a primeira na ordem do motor.

In [150]:
def simular_jogada(tabuleiro, linha, coluna, jogador):
    simulacao = [fileira.copy() for fileira in tabuleiro]
    fazer_jogada(simulacao, linha, coluna, jogador)
    return simulacao


def avaliar_jogadas(tabuleiro, jogador):
    resultados = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):

        simulacao = simular_jogada(tabuleiro, linha, coluna, jogador)
        pontuacao = avaliar_tabuleiro(simulacao, jogador)

        resultados.append({
            "row": linha,
            "col": coluna,
            "score": pontuacao
        })

    return resultados

In [151]:
def escolher_melhor_jogada(tabuleiro, jogador):
    jogadas = avaliar_jogadas(tabuleiro, jogador)

    if not jogadas:
        return None

    melhor_jogada = jogadas[0]

    for jogada in jogadas:
        if jogada["score"] > melhor_jogada["score"]:
            melhor_jogada = jogada

    return melhor_jogada

### 3.3 Planejamento de respostas

O planejamento compara a pontuação imediata de cada jogada com a menor pontuação após uma resposta adversária. Quando o adversário passa a vez, mantém a pontuação imediata.

In [152]:
def planejar_jogadas(tabuleiro, jogador):
    analises = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):
        simulacao = simular_jogada(tabuleiro, linha, coluna, jogador)

        pontuacao_atual = avaliar_tabuleiro(simulacao, jogador)
        respostas = movimentos_validos(simulacao, -jogador)
        pior_pontuacao = pontuacao_atual

        if respostas:
            pontuacoes = []
            for linha_resposta, coluna_resposta in respostas:
                resposta = simular_jogada(
                    simulacao, linha_resposta, coluna_resposta, -jogador
                )
                pontuacoes.append(avaliar_tabuleiro(resposta, jogador))

            pior_pontuacao = min(pontuacoes)

        analises.append({
            "row": linha,
            "col": coluna,
            "score_now": pontuacao_atual,
            "worst_response": pior_pontuacao,
            "opponent_moves": len(respostas)
        })

    return analises

### Algoritmo Minimax

Avalia jogadas futuras considerando que ambos os jogadores
buscam obter o melhor resultado possível.

In [153]:
def minimax(tabuleiro, jogador_atual, jogador_raiz, profundidade,
            alpha=float("-inf"), beta=float("inf"), usar_poda=True):

    movimentos = movimentos_validos(tabuleiro, jogador_atual)

    if not movimentos:
        if not movimentos_validos(tabuleiro, -jogador_atual):
            pretas, brancas = contar_pecas(tabuleiro)
            diferenca = (pretas - brancas) * jogador_raiz

            if diferenca > 0:
                return 10000 + diferenca
            if diferenca < 0:
                return -10000 + diferenca
            return 0

        # Passagem de vez
        return minimax(
            tabuleiro, -jogador_atual, jogador_raiz,
            profundidade, alpha, beta, usar_poda
        )

    if profundidade <= 0:
        return avaliar_tabuleiro(tabuleiro, jogador_raiz)

    maximizando = jogador_atual == jogador_raiz
    melhor = float("-inf") if maximizando else float("inf")

    for linha, coluna in movimentos:
        simulacao = [l.copy() for l in tabuleiro]
        fazer_jogada(simulacao, linha, coluna, jogador_atual)

        pontuacao = minimax(
            simulacao, -jogador_atual, jogador_raiz,
            profundidade - 1, alpha, beta, usar_poda
        )

        if maximizando:
            melhor = max(melhor, pontuacao)
            alpha = max(alpha, melhor)
        else:
            melhor = min(melhor, pontuacao)
            beta = min(beta, melhor)

        # Poda alfa-beta
        if usar_poda and alpha >= beta:
            break

    return melhor

In [154]:
def escolher_jogada_minimax(tabuleiro, jogador,
                            profundidade=3, usar_poda=True):

    melhor_jogada = None
    melhor_pontuacao = float("-inf")

    for linha, coluna in movimentos_validos(tabuleiro, jogador):
        simulacao = [l.copy() for l in tabuleiro]
        fazer_jogada(simulacao, linha, coluna, jogador)

        pontuacao = minimax(
            simulacao,
            -jogador,
            jogador,
            profundidade - 1,
            alpha=melhor_pontuacao,
            usar_poda=usar_poda
        )

        if pontuacao > melhor_pontuacao:
            melhor_pontuacao = pontuacao
            melhor_jogada = {
                "row": linha,
                "col": coluna,
                "score": pontuacao
            }

    return melhor_jogada

## 4. Ferramentas do jogo

As sete ferramentas operam sobre o estado compartilhado e usam as regras e os cálculos das seções anteriores. As coordenadas começam em zero.

### 4.1 Tabuleiro, movimentos legais e situação da partida

In [155]:
from langchain_core.tools import tool


@tool
def get_board_state() -> dict:
    """Consulta o estado atual do tabuleiro de Othello."""

    tabuleiro = estado_jogo["tabuleiro"]
    pretas, brancas = contar_pecas(tabuleiro)

    return {
        "board": [linha.copy() for linha in tabuleiro],
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas
    }


@tool
def get_valid_moves() -> dict:
    """Consulta os movimentos válidos do jogador atual.
    As coordenadas de linha e coluna começam em zero.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "moves": []}

    movimentos = movimentos_validos(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "moves": [
            {"row": linha, "col": coluna}
            for linha, coluna in movimentos
        ]
    }

In [156]:
@tool
def get_game_status() -> dict:
    """Consulta o estado da partida de Othello, incluindo
    pontuação, jogador atual, encerramento e vencedor.
    """

    tabuleiro = estado_jogo["tabuleiro"]

    pretas, brancas = contar_pecas(tabuleiro)
    terminou = jogo_terminou(tabuleiro)

    return {
        "game_over": terminou,
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas,
        "total_moves": len(estado_jogo["historico"]),
        "winner": vencedor_por_placar(pretas, brancas) if terminou else None
    }

### 4.2 Recomendação, histórico e planejamento

In [157]:
@tool
def get_heuristic_move() -> dict:
    """Recomenda a melhor jogada pela heurística posicional,
    sem executar o movimento.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "current_player": None,
            "recommended_move": None
        }

    recomendacao = escolher_melhor_jogada(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "recommended_move": recomendacao
    }

In [158]:
@tool
def get_move_history() -> dict:
    """Consulta o histórico de jogadas realizadas na partida."""

    historico = estado_jogo["historico"]

    return {
        "total_moves": len(historico),
        "moves": [
            {
                "player": jogada["player"],
                "row": jogada["row"],
                "col": jogada["col"]
            }
            for jogada in historico
        ]
    }

In [159]:
@tool
def get_strategic_plan() -> dict:
    """Avalia as jogadas válidas considerando a pontuação
    posicional e a pior resposta possível do adversário.
    Não executa nenhuma jogada.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "planning": []}

    return {
        "current_player": jogador,
        "planning": planejar_jogadas(
            estado_jogo["tabuleiro"], jogador
        )
    }

### 4.3 Execução de jogadas

`make_move` aplica uma jogada legal, registra suas coordenadas e atualiza o turno.

In [160]:
@tool
def make_move(row: int, col: int) -> dict:
    """Executa uma jogada de Othello para o jogador atual.
    As coordenadas row e col devem estar entre 0 e 7.
    """

    tabuleiro = estado_jogo["tabuleiro"]
    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "success": False,
            "message": "A partida já terminou."
        }

    if not fazer_jogada(tabuleiro, row, col, jogador):
        return {
            "success": False,
            "message": "Jogada inválida.",
            "valid_moves": movimentos_validos(tabuleiro, jogador)
        }

    estado_jogo["historico"].append({
        "player": jogador,
        "row": row,
        "col": col
    })

    proximo = proximo_turno(tabuleiro, jogador)
    estado_jogo["jogador_atual"] = proximo

    return {
        "success": True,
        "message": "Jogada realizada com sucesso.",
        "next_player": proximo,
        "game_over": proximo is None
    }

4.4 - getminimaxmove()

In [161]:
@tool
def get_minimax_move() -> dict:
    """Recomenda uma jogada utilizando Minimax com
    profundidade 3, sem modificar o tabuleiro.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "current_player": None,
            "recommended_move": None
        }

    return {
        "current_player": jogador,
        "depth": 3,
        "recommended_move": escolher_jogada_minimax(
            estado_jogo["tabuleiro"], jogador
        )
    }

### 4.4 Registro das sete ferramentas

O modo operacional disponibiliza o registro completo à LLM e ao `ToolNode`. O modo estratégico usa as três ferramentas de observação e `make_move`.

In [162]:
ferramentas = {
    "get_board_state": get_board_state,
    "get_valid_moves": get_valid_moves,
    "get_game_status": get_game_status,
    "get_heuristic_move": get_heuristic_move,
    "get_move_history": get_move_history,
    "get_strategic_plan": get_strategic_plan,
    "make_move": make_move,
    "get_minimax_move": get_minimax_move,
}

### 5.1 Credenciais e modelo

A célula solicita a chave Groq se `GROQ_API_KEY` estiver ausente do ambiente.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

In [ ]:
from langchain_groq import ChatGroq
from pydantic import BaseModel, ConfigDict, Field

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


class DecisaoOthello(BaseModel):
    model_config = ConfigDict(extra="forbid")

    row: int = Field(description="Linha da jogada, entre 0 e 7")
    col: int = Field(description="Coluna da jogada, entre 0 e 7")
    justificativa: str = Field(description="Motivo estratégico resumido")


llm_decisor = llm.with_structured_output(
    DecisaoOthello,
    method="json_schema",
    strict=True
)

### 5.2 Modo estratégico

O grafo observa, planeja, solicita uma decisão estruturada à LLM, valida a jogada e a executa. O histórico e o planejamento entram no contexto por meio de funções Python.

In [ ]:
import json

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage
)


class EstadoAgente(MessagesState, total=False):
    observacao: dict
    historico: str
    planejamento: list
    decisao: dict
    origem_decisao: str
    erro_decisao: str
    jogada_escolhida: dict
    resultado: dict
    historico_inicial: int
    chamadas_ferramenta: int
    ferramentas_executadas: list


INSTRUCOES_ESTRATEGICAS = """Você joga Othello. Escolha uma das jogadas válidas fornecidas.
Considere os cantos, os pesos posicionais, a mobilidade e as respostas possíveis.
As coordenadas começam em zero. Retorne a decisão no formato estruturado solicitado."""


def observar(estado: EstadoAgente) -> dict:
    # O fluxo estratégico consulta as três ferramentas de observação.
    tabuleiro = ferramentas["get_board_state"].invoke({})
    movimentos = ferramentas["get_valid_moves"].invoke({})
    situacao = ferramentas["get_game_status"].invoke({})
    ferramentas_consultadas = [
        "get_board_state", "get_valid_moves", "get_game_status"
    ]

    if situacao["game_over"]:
        return {
            "observacao": {
                "board": tabuleiro["board"],
                "current_player": tabuleiro["current_player"],
                "moves": movimentos["moves"],
                "status": situacao
            },
            "historico": obter_historico(),
            "ferramentas_executadas": ferramentas_consultadas
        }

    jogador = tabuleiro["current_player"]
    if jogador not in (PRETO, BRANCO) or movimentos["current_player"] != jogador:
        raise RuntimeError("Estado inconsistente entre as ferramentas.")

    if not movimentos["moves"]:
        raise RuntimeError("O jogador atual não possui jogadas válidas.")

    return {
        "observacao": {
            "board": tabuleiro["board"],
            "current_player": jogador,
            "moves": movimentos["moves"],
            "status": situacao
        },
        "historico": obter_historico(),
        "ferramentas_executadas": ferramentas_consultadas
    }


def planejar(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    jogador = observacao["current_player"]

    if observacao["status"]["game_over"]:
        return {"planejamento": []}

    analises = planejar_jogadas(observacao["board"], jogador)
    if len(analises) != len(observacao["moves"]):
        raise RuntimeError("O planejamento não corresponde às jogadas válidas.")

    return {"planejamento": analises}


def decidir_estrategico(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    if observacao["status"]["game_over"]:
        return {"decisao": {}, "origem_decisao": "sem_jogada"}

    contexto = {
        "jogador": observacao["current_player"],
        "tabuleiro": observacao["board"],
        "jogadas_validas": observacao["moves"],
        "historico": estado["historico"],
        "planejamento": estado["planejamento"]
    }

    try:
        resposta = llm_decisor.invoke([
            SystemMessage(content=INSTRUCOES_ESTRATEGICAS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False))
        ])
        if not isinstance(resposta, DecisaoOthello):
            resposta = DecisaoOthello.model_validate(resposta)
        return {
            "decisao": resposta.model_dump(),
            "origem_decisao": "llm",
            "erro_decisao": ""
        }
    except Exception as erro:
        return {
            "decisao": {},
            "origem_decisao": "fallback_heuristico",
            "erro_decisao": type(erro).__name__
        }


def validar_decisao(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    if observacao["status"]["game_over"]:
        return {"jogada_escolhida": {}}

    legais = observacao["moves"]
    decisao = estado.get("decisao", {})
    escolhida = next(
        (
            movimento for movimento in legais
            if movimento["row"] == decisao.get("row")
            and movimento["col"] == decisao.get("col")
        ),
        None
    )

    if escolhida is None:
        planejamento = estado["planejamento"]
        if planejamento:
            melhor = max(
                planejamento,
                key=lambda item: (
                    item["worst_response"],
                    item["score_now"],
                    -item["row"],
                    -item["col"]
                )
            )
            escolhida = {"row": melhor["row"], "col": melhor["col"]}
        else:
            escolhida = legais[0]

        return {
            "jogada_escolhida": escolhida,
            "origem_decisao": "fallback_heuristico",
            "erro_decisao": estado.get("erro_decisao") or "jogada_invalida"
        }

    return {"jogada_escolhida": escolhida}


def executar_jogada(estado: EstadoAgente) -> dict:
    if estado["observacao"]["status"]["game_over"]:
        return {"resultado": {"success": True, "game_over": True}}

    jogada = estado["jogada_escolhida"]
    jogador = estado["observacao"]["current_player"]
    quantidade_anterior = len(estado_jogo["historico"])

    resultado = ferramentas["make_move"].invoke({
        "row": jogada["row"],
        "col": jogada["col"]
    })

    if not resultado["success"]:
        raise RuntimeError(resultado["message"])

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O turno deve registrar exatamente uma jogada.")

    registrada = estado_jogo["historico"][-1]
    if (
        registrada["player"] != jogador
        or registrada["row"] != jogada["row"]
        or registrada["col"] != jogada["col"]
    ):
        raise RuntimeError("A jogada registrada não corresponde à decisão.")

    usadas = estado.get("ferramentas_executadas", []) + ["make_move"]
    return {"resultado": resultado, "ferramentas_executadas": usadas}


# Modo 1: o grafo controla a ordem completa das etapas.
grafo_estrategico = StateGraph(EstadoAgente)
grafo_estrategico.add_node("observar", observar)
grafo_estrategico.add_node("planejar", planejar)
grafo_estrategico.add_node("decidir", decidir_estrategico)
grafo_estrategico.add_node("validar", validar_decisao)
grafo_estrategico.add_node("executar", executar_jogada)

grafo_estrategico.add_edge(START, "observar")
grafo_estrategico.add_edge("observar", "planejar")
grafo_estrategico.add_edge("planejar", "decidir")
grafo_estrategico.add_edge("decidir", "validar")
grafo_estrategico.add_edge("validar", "executar")
grafo_estrategico.add_edge("executar", END)

app_estrategico = grafo_estrategico.compile()

### 5.3 Modo operacional

A LLM escolhe as ferramentas após receber o contexto inicial. O grafo aceita uma chamada por resposta e permite até seis chamadas por turno. Uma jogada válida encerra o turno. Os dois grafos usam a mesma validação e execução em caso de decisão inválida.

In [ ]:
INSTRUCOES_OPERACIONAIS = """Você joga Othello e pode escolher quando usar as ferramentas.
Use as informações de memória e planejamento recebidas para avaliar a jogada.
As ferramentas get_board_state, get_valid_moves e get_game_status consultam o ambiente.
A ferramenta make_move executa uma jogada para o jogador atual; nunca escolha o adversário.
As coordenadas começam em zero. Faça uma chamada de ferramenta por resposta e pare
depois que uma chamada make_move for executada com sucesso. Não altere o tabuleiro
diretamente. Se uma ferramenta retornar erro, corrija a solicitação ou use as
informações recebidas. O sistema valida cada jogada e encerra o turno após um movimento."""


llm_com_ferramentas = llm.bind_tools(list(ferramentas.values()))
MAX_FERRAMENTAS_POR_TURNO = 6
no_ferramentas = ToolNode(
    list(ferramentas.values()),
    handle_tool_errors=True
)


def preparar_operacional(estado: EstadoAgente) -> dict:
    tabuleiro = [linha.copy() for linha in estado_jogo["tabuleiro"]]
    jogador = estado_jogo["jogador_atual"]
    movimentos = movimentos_validos(tabuleiro, jogador)

    if jogador not in (PRETO, BRANCO) or not movimentos:
        raise RuntimeError("Não há um turno válido para o agente.")

    jogadas = [
        {"row": linha, "col": coluna}
        for linha, coluna in movimentos
    ]
    analises = planejar_jogadas(tabuleiro, jogador)
    historico = obter_historico()
    contexto = {
        "jogador_atual": jogador,
        "tabuleiro": tabuleiro,
        "jogadas_validas": jogadas,
        "historico": historico,
        "planejamento": analises
    }

    return {
        "observacao": {
            "board": tabuleiro,
            "current_player": jogador,
            "moves": jogadas,
            "status": {"game_over": False}
        },
        "historico": historico,
        "planejamento": analises,
        "historico_inicial": len(estado_jogo["historico"]),
        "chamadas_ferramenta": 0,
        "erro_decisao": "",
        "ferramentas_executadas": [],
        "messages": [
            SystemMessage(content=INSTRUCOES_OPERACIONAIS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False))
        ]
    }


def decidir_operacional(estado: EstadoAgente) -> dict:
    try:
        resposta = llm_com_ferramentas.invoke(estado["messages"])
    except Exception as erro:
        return {
            "erro_decisao": type(erro).__name__,
            "origem_decisao": "fallback_heuristico"
        }

    chamadas = resposta.tool_calls or []
    total = estado.get("chamadas_ferramenta", 0) + len(chamadas)
    erro_chamada = ""

    if not chamadas:
        erro_chamada = "resposta_sem_chamada_de_ferramenta"
    elif len(chamadas) != 1:
        erro_chamada = "multiplas_chamadas_de_ferramenta"
    elif chamadas[0]["name"] not in ferramentas:
        erro_chamada = "ferramenta_desconhecida"
    elif total > MAX_FERRAMENTAS_POR_TURNO:
        erro_chamada = "limite_de_ferramentas"

    return {
        "messages": [resposta],
        "chamadas_ferramenta": total,
        "erro_decisao": erro_chamada,
        "origem_decisao": (
            "fallback_heuristico" if erro_chamada else ""
        )
    }


def rotear_operacional(estado: EstadoAgente) -> str:
    if len(estado_jogo["historico"]) > estado["historico_inicial"]:
        return END
    if estado.get("erro_decisao"):
        return "fallback"
    return "tools"


def rotear_apos_ferramenta(estado: EstadoAgente) -> str:
    if len(estado_jogo["historico"]) > estado["historico_inicial"]:
        return END
    if estado["chamadas_ferramenta"] >= MAX_FERRAMENTAS_POR_TURNO:
        return "fallback"
    return "decidir"


def fallback_operacional(estado: EstadoAgente) -> dict:
    return {
        "decisao": {},
        "origem_decisao": "fallback_heuristico",
        "erro_decisao": (
            estado.get("erro_decisao") or "limite_de_ferramentas"
        )
    }


# Modo 2: a LLM escolhe as ferramentas; o grafo valida e limita o ciclo.
grafo_operacional = StateGraph(EstadoAgente)
grafo_operacional.add_node("preparar", preparar_operacional)
grafo_operacional.add_node("decidir", decidir_operacional)
grafo_operacional.add_node("tools", no_ferramentas)
grafo_operacional.add_node("fallback", fallback_operacional)
grafo_operacional.add_node("validar", validar_decisao)
grafo_operacional.add_node("executar", executar_jogada)

grafo_operacional.add_edge(START, "preparar")
grafo_operacional.add_edge("preparar", "decidir")
grafo_operacional.add_conditional_edges(
    "decidir",
    rotear_operacional,
    {"tools": "tools", "fallback": "fallback", END: END}
)
grafo_operacional.add_conditional_edges(
    "tools",
    rotear_apos_ferramenta,
    {"decidir": "decidir", "fallback": "fallback", END: END}
)
grafo_operacional.add_edge("fallback", "validar")
grafo_operacional.add_edge("validar", "executar")
grafo_operacional.add_edge("executar", END)

app_operacional = grafo_operacional.compile()

### 5.4 Visualização dos fluxos

A célula exibe a definição Mermaid dos dois grafos.

In [ ]:
print("Modo estratégico:")
print(app_estrategico.get_graph().draw_mermaid())
print("\nModo operacional:")
print(app_operacional.get_graph().draw_mermaid())

### 5.5 Seleção do modo e execução de um turno

Escolha **1** para o modo estratégico ou **2** para o operacional. Cada turno registra a origem da decisão e as ferramentas solicitadas e executadas.

In [ ]:
escolha = input(
    "Modo de autonomia (1 = estratégica, 2 = operacional): "
).strip()

if escolha not in ("1", "2"):
    raise ValueError("Escolha 1 (estratégica) ou 2 (operacional).")

MODO_AUTONOMIA = {
    "1": "estrategica",
    "2": "operacional"
}[escolha]

print("Modo selecionado:", MODO_AUTONOMIA)

In [ ]:
def executar_turno_agente():
    if estado_jogo["jogador_atual"] is None:
        return None

    quantidade_anterior = len(estado_jogo["historico"])

    if MODO_AUTONOMIA == "estrategica":
        resultado = app_estrategico.invoke({"messages": []})
        solicitadas = resultado.get("ferramentas_solicitadas", [])
        executadas = resultado.get("ferramentas_executadas", [])
    else:
        resultado = app_operacional.invoke(
            {"messages": []},
            config={"recursion_limit": 20}
        )
        mensagens = resultado.get("messages", [])
        solicitadas = [
            chamada["name"]
            for mensagem in mensagens
            if isinstance(mensagem, AIMessage)
            for chamada in mensagem.tool_calls
        ]
        executadas = [
            mensagem.name
            for mensagem in mensagens
            if isinstance(mensagem, ToolMessage) and mensagem.name
        ]

        if (
            len(estado_jogo["historico"]) == quantidade_anterior + 1
            and not resultado.get("origem_decisao")
        ):
            resultado["origem_decisao"] = "llm_tool_call"

        if resultado.get("origem_decisao") == "fallback_heuristico":
            executadas.append("make_move")

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O agente deve executar exatamente uma jogada.")

    jogada = estado_jogo["historico"][-1]
    origem = resultado.get("origem_decisao", "llm")
    jogada.update({
        "modo_autonomia": MODO_AUTONOMIA,
        "origem_decisao": origem,
        "ferramentas_solicitadas": solicitadas,
        "ferramentas_executadas": executadas
    })
    if resultado.get("erro_decisao"):
        jogada["erro_decisao"] = resultado["erro_decisao"]

    resultado["ferramentas_solicitadas"] = solicitadas
    resultado["ferramentas_executadas"] = executadas
    print("Modo:", MODO_AUTONOMIA)
    print("Origem da decisão:", origem)
    print("Jogada:", (jogada["row"], jogada["col"]))
    print("Ferramentas solicitadas:", solicitadas)
    print("Ferramentas executadas:", executadas)

    return resultado

## 6. Partidas e exemplos

`jogar_partida_heuristica()` coloca dois jogadores heurísticos em uma partida.

In [ ]:
def jogar_partida_heuristica():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        tabuleiro = estado_jogo["tabuleiro"]
        jogador = estado_jogo["jogador_atual"]
        jogada = escolher_melhor_jogada(tabuleiro, jogador)

        if jogada is None:
            raise ValueError("Jogador atual sem movimentos válidos.")

        resultado = make_move.invoke({
            "row": jogada["row"],
            "col": jogada["col"]
        })

        if not resultado["success"]:
            raise RuntimeError(resultado["message"])

    return get_game_status.invoke({})

### 6.1 Partida entre a LLM e o agente heurístico

`jogar_partida_completa()` usa o modo selecionado para as pretas e a heurística para as brancas. Retorna a situação final e o histórico.

In [ ]:
def jogar_partida_completa():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        jogador = estado_jogo["jogador_atual"]
        quantidade_anterior = len(estado_jogo["historico"])

        if jogador == PRETO:
            executar_turno_agente()
        else:
            jogada = escolher_melhor_jogada(
                estado_jogo["tabuleiro"], jogador
            )
            if jogada is None:
                raise ValueError("Jogador atual sem movimentos válidos.")

            resultado = make_move.invoke({
                "row": jogada["row"],
                "col": jogada["col"]
            })
            if not resultado["success"]:
                raise RuntimeError(resultado["message"])

            estado_jogo["historico"][-1].update({
                "modo_autonomia": "heuristica",
                "origem_decisao": "heuristica",
                "ferramentas_solicitadas": [],
                "ferramentas_executadas": ["make_move"]
            })

        if len(estado_jogo["historico"]) != quantidade_anterior + 1:
            raise RuntimeError("Cada turno deve executar uma jogada.")

    return {
        "status": get_game_status.invoke({}),
        "historico": [
            jogada.copy() for jogada in estado_jogo["historico"]
        ]
    }

### 6.2 Exemplos para execução manual

Consulte o plano do turno atual com `get_strategic_plan.invoke({})`.

Para iniciar uma partida nova e observar um turno da LLM, execute:

```python
estado_jogo = iniciar_partida()
resultado = executar_turno_agente()
print(estado_jogo["historico"][-1])
```